In [ ]:
import numpy as np
import pandas as pd

from acise_p1_cigp.data.load import (
    LABEL_COL,
    PASS_LABEL,
    Dataset,
    load_project,
    preprocess,
)
from acise_p1_cigp.viz import plots

raw = load_project(all=True)
print(raw.shape, raw["project"].nunique(), "projects")

In [ ]:
empty = raw.groupby("project").apply(lambda df: df.columns[df.isna().all()].tolist())
empty = empty[empty.str.len() > 0]
print(f"projects with all-NaN columns: {len(empty)}")
if len(empty):
    display(empty.to_frame("all_nan_columns"))

In [ ]:
def profile(df: pd.DataFrame) -> pd.DataFrame:
    num = df.select_dtypes(include=np.number)

    out = num.describe(percentiles=[0.05, .25, .5, .75, .95]).T
    out["dtype"] = num.dtypes.astype(str)
    out["nan_%"] = num.isna().mean() * 100
    out["zero_%"] = (num == 0).mean() * 100
    out["nunique"] = num.nunique()
    out["skew"] = num.skew()
    
    return out.round(4)

# Raw Data Anaylsis - before preprocessing

In [ ]:
raw_feats = raw.drop(columns=["project"])

display(profile(raw_feats))
display(raw_feats.sample(10, random_state=42).T)
display(raw_feats.dtypes.value_counts().to_frame("n_features"))
display(
    raw.groupby("project")[LABEL_COL]
    .agg(builds="size", pass_rate=lambda s: (s == PASS_LABEL).mean())
    .sort_values("builds", ascending=False)
)

# Stateless Preprocessing

In [ ]:
def frame(dt: Dataset) -> pd.DataFrame:
    return dt.X.assign(failed=dt.y, project=dt.groups)


dt1 = preprocess(raw, target="T1")
dt2 = preprocess(raw, target="T2")

t1, t2 = frame(dt1), frame(dt2)

print("T1", dt1.X.shape, dt1.meta)
print("T2", dt2.X.shape, dt2.meta)
print("dropped (T1):", sorted(set(raw_feats.columns) - set(dt1.X.columns)))
print("dropped (T2):", sorted(set(raw_feats.columns) - set(dt2.X.columns)))

In [ ]:
display(profile(dt1.X))
display(dt1.X.sample(10, random_state=42).T)

In [ ]:
display(profile(dt2.X))
display(dt2.X.sample(10, random_state=42).T)

In [ ]:
balance = (
    pd.DataFrame(
        {
            "builds_raw": raw.groupby("project").size(),
            "builds_T1": dt1.groups.value_counts(),
            "fail_%_T1": dt1.y.groupby(dt1.groups).mean() * 100,
            "builds_T2": dt2.groups.value_counts(),
            "first_fails_T2": dt2.y.groupby(dt2.groups).sum(),
            "first_fail_%_T2": dt2.y.groupby(dt2.groups).mean() * 100,
            "dev_rows_T2": pd.Series(dt2.dev_mask).groupby(dt2.groups.to_numpy()).sum(),
        }
    )
    .round(4)
    .sort_values("builds_raw", ascending=False)
)
display(balance)

med = pd.DataFrame({"median_T1": dt1.X.median(), "median_T2": dt2.X.median()})
med["rel_med_diff"] = (med.median_T2 - med.median_T1) / med.median_T1.abs().replace(0, np.nan)
display(med.sort_values("rel_med_diff", key=abs, ascending=False).head(10))

# Visualization

## Label balance per project

In [ ]:
for tag, df in (("T1", t1), ("T2", t2)):
    d = df.assign(label=df.failed.map({0: "pass", 1: "fail"}))
    fig = plots.label_balance(d, project="project", label="label")
    plots.save(fig.update_layout(title=f"Label balance ({tag})"), f"eda_label_balance_{tag}")
    fig.show()

## Fail streak length, chronological per project

In [ ]:
def fail_streaks(y: pd.Series) -> pd.Series:
    run = (y != y.shift()).cumsum()
    g = y.groupby(run).agg(["first", "size"])
    return g.loc[g["first"] == 1, "size"]

streaks = pd.concat([fail_streaks(y) for _, y in dt1.y.groupby(dt1.groups)], ignore_index=True)
display(streaks.describe().round(4))
print(f"P(streak=1) = {round((streaks == 1).mean(), 4)}")

fig = plots.streak_hist(streaks, nbins=40, log_y=True)
plots.save(fig.update_layout(title="Fail streaks (T1)"), "eda_fail_streaks_T1")
fig.show()

## Top features by label (T2, log axis)

In [ ]:
# nominal codes have no order, so Spearman is meaningless for them
NOMINAL = ["change_type", "action_changed", "day_of_week", "time_of_day"]
cmany = [c for c in dt2.X.columns if dt2.X[c].nunique() > 2 and c not in NOMINAL]
rank = dt2.X[cmany].corrwith(dt2.y, method="spearman").abs().sort_values(ascending=False)
top = rank.head(20).index.tolist()
display(rank.head(20).to_frame("spearman_corr").round(4))

d = t2.assign(label=t2.failed.map({0: "pass", 1: "fail"}))
fig = plots.dist_by_label(d, top, label="label", kind="box", log_y=True)  # zero vanish on log axis
plots.save(fig.update_layout(height=1200, title="Top features by label (T2, log axis)"), "eda_top_features_T2")
fig.show()

## Spearman correlation of the top-30 features (T2)

In [ ]:
corr = dt2.X[rank.head(30).index].corr(method="spearman")
fig = plots.corr_heatmap(corr)
plots.save(fig.update_layout(height=700, title="Spearman correlation of the top-30 features (T2)"), "eda_corr_top30_T2")
fig.show()

## Near-duplicate pairs

Used for redundancy pruning.

In [ ]:
c = dt2.X.corr(method="spearman").abs()
pairs = c.where(np.triu(np.ones(c.shape, bool), 1)).stack()
display(pairs[pairs > 0.95].sort_values(ascending=False).head(50).to_frame("spearman_corr"))